In [1]:
from os import getenv
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI(
    base_url="https://26120057-2478-resource.openai.azure.com/openai/v1",
    api_key=getenv("AZURE_API_KEY"),
)

MODEL = "gpt-5-nano"

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

Here’s how to run Ollama locally:

- Install Ollama (choose your OS from the download page):
  - macOS: download the .pkg and install it.
  - Windows: download the .msi and install it.
  - Linux: run this in the terminal:
    curl -fsSL https://ollama.com/install.sh | sh

- After installation, start the model locally:
  ollama run llama3
  - This downloads the LLaMA 3 model (~4GB), starts the model locally, and opens a chat-like interface.

- Test that the local server is running:
  curl http://localhost:11434
  You should see a JSON response with model information.

- Optional: use the Python client
  pip install ollama

  Minimal Python example:
  import ollama

  response = ollama.chat(
      model='llama3',
      messages=[{"role": "user", "content": your_prompt}]
  )

  print(response['message']['content'])

Troubleshooting (if you see a connection refused error):
- Restart the Ollama server:
  !nohup ollama serve > nohup.out 2>&1 &
- Rerun the cell that starts ollama serve.


In [5]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model=MODEL,
    input=messages,
)

response.output_text

'Yes—usually you can join, but the exact steps depend on the course and platform. To help you quickly, tell me:\n\n- The course name\n- The platform (Coursera, edX, Udemy, a university site, etc.)\n- Whether you want a certificate or just access to the materials\n- If you know the start date or if it’s self-paced\n\nIf you don’t have that yet, here are typical paths on common platforms:\n\n- Coursera\n  - Log in, search for the course, click Enroll.\n  - Choose “Audit the course” for free access to materials (no certificate) or enroll for full access/certificates.\n  - Check if the course has a fixed start date or is ongoing.\n  - If you miss enrollment, some courses still let you audit even after the start date.\n\n- edX\n  - Find the course, click Enroll.\n  - You can often choose “Audit this course” (free) or pay for a Verified Certificate.\n  - Note start dates for scheduled courses.\n\n- Udemy\n  - Courses are typically purchasable anytime; add to cart and buy for immediate access

In [6]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [7]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [8]:
response = openai_client.responses.create(
    model=MODEL,
    input=messages,
    tools=[search_tool]
)

In [9]:
function_calls = [item for item in response.output if item.type == 'function_call']
len(function_calls)

1

In [10]:
call = function_calls[0]

In [11]:
call

ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it starts"}', call_id='call_18rzMZXHSf3WCxvk2upBkdWu', name='search', type='function_call', id='fc_06f71eb1f4b06340006aef7918b5b88194a18eebd518bd5dea', async_=None, caller=None, namespace=None, status='completed')

In [12]:
import json

args = json.loads(call.arguments)
args

{'query': 'Can I join the course after it starts'}

In [13]:
call.name

'search'

In [14]:
results = search(**args)

In [15]:
result_json = json.dumps(results, indent=2)

In [16]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
}

In [17]:
messages.extend(response.output)

In [18]:
messages.append(function_call_output)

In [19]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseReasoningItem(id='rs_06f71eb1f4b06340006aef7916c17c8194a392dda62f144df0', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABq73kZCScV8B40TfPzBpzGcJaVzQmWrRR57EzKGjDAUHwfB9XhweY_ccfeo-s2b4JoayKC1l1f5rj48Xr2iv6krW1s525935XES8pX_raW5M8qfxuUm-nPWvyyZpFq6tcifX3pd1Gm8NJLp1GhfFpF7M24x_ABPlM8ULmgRHjOvmmAwDA8Kavq3Sv_dqk5xZ7o-fi0_qmV81Wy07awrAApRdbiXGDOlAAn3J_yVIU1U3l0aaIyYb-9g-BAjnz8lzA6M32nUBznV8pLk4cfc0PZECDql9eiO35MasgpOApLX3Wf2gkA0Pb2PZuq4IiIrWQBzeMN93lJDSIy-QOdLdsy4b9v0DYOt8OMJQvtQAhHFzQvOXj4OJSxmKzN1sZuzxfXkUy93J2i9dePFKKDl313nedCTM15Yinz1n86eRCwx1F953pZZRpLL2e7_R-a-WrtgKO0QZD1zyx5L4BAeBOcJwI00o0dk0SNnUpyMkni4VfPd1R83HjJgZLWneY4-y_2LUA-i5bmZ-uGoz8RKi_2I6WnfR3xPISSfGyaOROW3sIGoJP-sYhZP_mv9mPU5PNs7VRhUp-XEeY4F1cU35vqzsCK3-k9Q-FDe-WsSIuAbUU4UhRxppsbTWkW3Vzhmvr_AmVHGFn61RPL5Q3tqcVl06P6D82aWNnrmatHMHQmtJsyv69SJ82Tpa023v83nw5XGzJR-SClTTqyqRu9dYnTBRXtQ8gu7a6jG5q7XOMa8Y6zS9KSU7dHaRU4RB3gkO89GV

In [20]:
response = openai_client.responses.create(
    model=MODEL,
    input=messages,
    tools=[search_tool]
)

In [21]:
print(response.output_text)

Yes. You can still join the LLM Zoomcamp.

- You’re accepted automatically—you don’t need a confirmation email to start.
- You can start learning and submit homework (while the submission form is open).
- If you want to receive the certificate, you’ll need to submit your capstone project while submissions are open.

Would you like me to pull up the course page or give you the direct link to get started?


In [22]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(1075, 1071)

In [23]:
def calculate_gpt5nano_price(input_tokens, output_tokens):
    # Prices per 1M tokens (gpt-5-nano)
    INPUT_PRICE_PER_MILLION = 0.05   # $0.05 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.40  # $0.40 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt5nano_price(usage.input_tokens, usage.output_tokens)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.00048215


In [24]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]


In [25]:
response = openai_client.responses.create(
    model=MODEL,
    input=messages,
    tools=[search_tool]
)

response.output_text
print(response.output_text)

In [26]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [27]:
print([item for item in response.output if item.type == 'function_call'])

[ResponseFunctionToolCall(arguments='{"query":"late enrollment policy join course FAQ"}', call_id='call_C19nc1DgaHj0Lk5AJejCM0mj', name='search', type='function_call', id='fc_04db72b042c8004e006aef79247ef48196b5f64150843ef854', async_=None, caller=None, namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"how to enroll in course FAQ"}', call_id='call_KIThCRrRS7fUcfaM88y2HmG2', name='search', type='function_call', id='fc_04db72b042c8004e006aef79247f1081968ec51ff194205bb8', async_=None, caller=None, namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"course waitlist enrollment"}', call_id='call_TcNkIaQDolOTQ7WEf2PQn913', name='search', type='function_call', id='fc_04db72b042c8004e006aef792482148196999e9f05d257dcf9', async_=None, caller=None, namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"access course materials after enrolling"}', call_id='call_q8XGu1pju9r9s8GJM8GskIQ2', name='search', type='

In [28]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

function_call: search {"query":"late enrollment policy join course FAQ"}
function_call: search {"query":"how to enroll in course FAQ"}
function_call: search {"query":"course waitlist enrollment"}
function_call: search {"query":"access course materials after enrolling"}
function_call: search {"query":"self-paced course enrollment open"}


In [29]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseReasoningItem(id='rs_04db72b042c8004e006aef7920fb908196a57013381e1c9a9c', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABq73kkavELHum2EMiqA_LGluUrCnZ_5Tp2_s1kKr6pnEPk9TlEp-CsDmw9jTQ12MnsTS9VIEzBva7NR3jF5LMbFTx6IoITqMIRwh8A1EswLHtidbSHyIndneuA4ukD_6WxMZwL6K8Gdt7jmqKg9P0Kq0uafQYoAzghI7pqAvibPChO6H31QwBfumu5pBXg1ZRVUsHNbunikJ7v_fmyPA3z7INg8ho-AgZrLn3nGX4waiQQgMDayuAIb_oswHQELkEN1Xjyb

In [30]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it= 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model=MODEL,
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True
        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)

    it += 1

    if has_function_calls == False:
        break


iteration #1...
function_call: search {"query":"How to join a course enrollment open enrollment can I join FAQ"}
function_call: search {"query":"Enroll in course FAQ access join course is it free paid path"}
function_call: search {"query":"Can I join the course? enrollment requirements prerequisites"}
function_call: search {"query":"Auditing a course join waitlist FAQ university platform"}
function_call: search {"query":"Course onboarding how to enroll"}
iteration #2...
ASSISTANT:
Yes—you can join. If you want to receive a certificate, you’ll need to submit your capstone project while submissions are still open.

Helpful next steps:
- Start now by checking the LLM Zoomcamp docs and GitHub repository to get oriented:
  - Docs: https://datatalks.club/docs/courses/llm-zoomcamp/
  - GitHub: https://github.com/DataTalksClub/llm-zoomcamp
- See deadlines and enrollments on the course management platform (for the current cohort): https://courses.datatalks.club/llm-zoomcamp-2026/
- Typical work

In [31]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searchers.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [32]:
def agent_loop(instructions, question, model="gpt-5-nano") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [33]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama local run"}
function_call: search {"query":"Olama installation"}
function_call: search {"query":"Olama run locally"}
function_call: search {"query":"Olama tutorial"}
function_call: search {"query":"Olama development environment"}
iteration #2...
ASSISTANT:
Do you mean Ollama (the local LLM server) or something named “Olama”? If you meant Ollama, here’s how to run it locally and use it from the course setup.

What Ollama is used for in this course
- The course supports running local models via Ollama, so you can swap in a local model instead of a cloud provider.
- If you run Ollama locally, point the AI provider (OpenAI-compatible) to your local Ollama endpoint (usually http://localhost:11434).

How to run Ollama locally (quickstart)
1) Install Ollama
- macOS: follow steps on ollama.ai (often via the installer or Homebrew)
- Windows: download the Windows installer from ollama.ai
- Linux: follow the Linux instructions on ollama.ai
2) 

'Do you mean Ollama (the local LLM server) or something named “Olama”? If you meant Ollama, here’s how to run it locally and use it from the course setup.\n\nWhat Ollama is used for in this course\n- The course supports running local models via Ollama, so you can swap in a local model instead of a cloud provider.\n- If you run Ollama locally, point the AI provider (OpenAI-compatible) to your local Ollama endpoint (usually http://localhost:11434).\n\nHow to run Ollama locally (quickstart)\n1) Install Ollama\n- macOS: follow steps on ollama.ai (often via the installer or Homebrew)\n- Windows: download the Windows installer from ollama.ai\n- Linux: follow the Linux instructions on ollama.ai\n2) Start the local API\n- Start the Ollama REST API (the exact command may vary by version):\n  - ollama serve\n  - or ollama start (check the docs if your version uses a different command)\n3) Pull a model (choose something light for development)\n- ollama pull llama-7b-chat\n- or any small model you

In [34]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"Queen's Gambit chess opening"}
function_call: search {"query":"Queen's Gambit Accepted"}
function_call: search {"query":"Queen's Gambit Declined"}
iteration #2...
ASSISTANT:
The Queen's Gambit is one of the oldest and most respected chess openings. It starts with the moves:

- 1. d4 d5 2. c4

White offers the pawn on c4 to Black in exchange for better control of the center (the d5 and e4 squares) and faster development.

Key ideas
- If Black accepts (2...dxc4), White recovers the pawn later and gains central influence and active piece play.
- If Black declines (common replies are 2...e6 or 2...c6), White keeps pressure in the center and often plays to build a strong pawn center with e4, or to pressure d5 from move order.

 Main branches
- Queen's Gambit Declined (QGD): 1.d4 d5 2.c4 e6. Very solid for Black; White usually develops Nf3, e3, and Bg2 or Bd3 and aims for central space and quick development.
- Queen's Gambit Accepted (QGA): 1.d

"The Queen's Gambit is one of the oldest and most respected chess openings. It starts with the moves:\n\n- 1. d4 d5 2. c4\n\nWhite offers the pawn on c4 to Black in exchange for better control of the center (the d5 and e4 squares) and faster development.\n\nKey ideas\n- If Black accepts (2...dxc4), White recovers the pawn later and gains central influence and active piece play.\n- If Black declines (common replies are 2...e6 or 2...c6), White keeps pressure in the center and often plays to build a strong pawn center with e4, or to pressure d5 from move order.\n\n Main branches\n- Queen's Gambit Declined (QGD): 1.d4 d5 2.c4 e6. Very solid for Black; White usually develops Nf3, e3, and Bg2 or Bd3 and aims for central space and quick development.\n- Queen's Gambit Accepted (QGA): 1.d4 d5 2.c4 dxc4. Black takes the pawn; White continues with moves like Nf3 and e3 to regain the pawn and seize central influence.\n- Slav Defense (often reached via 2...c6): 1.d4 d5 2.c4 c6. A very popular, sol

In [35]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
function_call: search {"query":"queen's gambit"}
function_call: search {"query":"queen gambit FAQ"}
function_call: search {"query":"chess opening queen's gambit"}
iteration #2...
ASSISTANT:
I couldn’t find any FAQ entry about Queen’s Gambit in this course’s FAQ database. It seems to be outside the scope of our course topics (which cover course logistics and related material like RAG, tool usage, etc.).

If you meant a chess opening (Queen’s Gambit), that topic isn’t covered in the FAQ. I can:
- Help with any course logistics questions (deadlines, office hours, grading, etc.).
- Look up FAQ entries related to other course topics you’re studying.
- Perform a broader search for related course content if you’d like.

Would you like me to search for another course-related topic or assist with something else in the course?


'I couldn’t find any FAQ entry about Queen’s Gambit in this course’s FAQ database. It seems to be outside the scope of our course topics (which cover course logistics and related material like RAG, tool usage, etc.).\n\nIf you meant a chess opening (Queen’s Gambit), that topic isn’t covered in the FAQ. I can:\n- Help with any course logistics questions (deadlines, office hours, grading, etc.).\n- Look up FAQ entries related to other course topics you’re studying.\n- Perform a broader search for related course content if you’d like.\n\nWould you like me to search for another course-related topic or assist with something else in the course?'

In [36]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [37]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [38]:
search_tool

{'type': 'function',
 'name': 'search',
 'description': 'Search the FAQ database for entries matching the given query.',
 'parameters': {'type': 'object',
  'properties': {'query': {'type': 'string',
    'description': 'Search query text to look up in the course FAQ.'}},
  'required': ['query'],
  'additionalProperties': False}}

In [39]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [40]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [41]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course FAQ.'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [46]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [50]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

The search is keyword-based: it only matches exact words. Before searching,
fix any spelling mistakes in the user's question (e.g. "Olama" -> "Ollama")
and use the correct names of tools and technologies in your queries.
If the results don't match what the user asked about, search again with
corrected spellings, synonyms, or alternative keywords.

Make multiple searches. First perform search, analyze the results
and then perform more searches.

The question has to be about the course or its logistics, offtopic questions
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [ ]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5-nano", client=openai_client)
)

In [ ]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [54]:
result.cost
result.all_messages

[EasyInputMessage(content='You\'re a course teaching assistant.\nYou\'re given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function.\nUse as many keywords from the user question as possible when making first requests.\n\nThe search is keyword-based: it only matches exact words. Before searching,\nfix any spelling mistakes in the user\'s question (e.g. "Olama" -> "Ollama")\nand use the correct names of tools and technologies in your queries.\nIf the results don\'t match what the user asked about, search again with\ncorrected spellings, synonyms, or alternative keywords.\n\nMake multiple searches. First perform search, analyze the results\nand then perform more searches.\n\nThe question has to be about the course or its logistics, offtopic questions\nshouldn\'t be answered. If the search returns nothing, it\'s likely an off-topic question.\nIf you can\'t answer the question using FAQ, don\'t do it yourself. Only us

In [ ]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)